In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (average_precision_score, precision_score, recall_score, f1_score,
classification_report, confusion_matrix, ConfusionMatrixDisplay)

from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

# **Part A — Frame the problem before you look at any data**
## **A1 — The framing template**

**Problem:-**
* The bank wants to identify customers who are more likely to subscribe to a term deposit during its marketing campaigns.

**Decision:-**
* The marketing team can use the predicted scores to prioritize customers for outreach, subject to available staff and campaign capacity.

**User:-**
* The bank's direct-marketing team.

**Target:-**
* The y column, where yes means that the customer subscribed to a term deposit and no means that the customer did not subscribe.

**Unit:-**
* One row represents one recorded customer contact in the bank marketing dataset. The file does not provide a customer ID that would allow us to verify whether the same person appears in multiple records.

**Metric:-**
* Average Precision (AP). The positive class is relatively uncommon, so AP measures the quality of the ranking of likely subscribers without relying on accuracy alone.

**Baseline:-**
* A Constant-score ranking that assigns the same score to every customer. Its expected AP is approximately the positive-class prevalence, around 0.12 in the full dataset.
* The actual holdout baseline will be measured after splitting the data.

**Good Enough:-**
* As a provisional project target, achieve an AP at least 1.5 times the constant-score baseline on the chronological holdout.
* This is a project hypothesis, not a proven business threshold.
* Its usefullness must be checked again campaign costs, outreach capacity and the value of successful subscriptions.

**Failure:-**
* The project will not meet its objective if the model can't improve AP over the constant-score baseline on the holdout, or if its apparent performance depends on information that would not be available when the marketing team must decide whom to contact.

**False Positive Cost:-**
* A customer is prioritized but does not subscribe.
* This can waste staff time and marketing resources.

**False Negative Cost:-**
* A customer who would subscribe is not prioritized.
* This can mean a missed business opportunity.



This exact financial cost of either error is not provided by this dataset, so these costs can't yet be quantified ralibly.

## **A2 — Choose your metric from the decision, not from habit**

The target has two classes: `yes` or `no`. If relatively few clients subscribe, accuracy could look high even when a rule rarely identifies subcribers correctly.  
Average Precision evaluates the quality of a ranking by summarizing precision across different recall levels. It is useful when the team wants to prioritize a limited list of customers.

## **A3 — Decide your baseline now, not later**

Our baseline assigns the same score to every customer: the positive class proportion observed in the training set. It uses no customer-specific information.  
For an uninformative constant ranking, expected AP is approximately the proportion of positive examples.  
For example, if 12% of clients subscribed, the expected baseline AP would be about 0.12.

**Expected Baseline Score:-**
* The positive class rate calculated in A2, approximately.

**Good Enough Score:-**
* Provisional AP of 0.20 or higher, with a meaningful improvement over the measured baseline.

# **Part B — Interrogate the dataset before you commit to it**
## **B1 — Dataset selection**

**Dataset :-** Bank Marketing 
* The commonly distributed UCI Bank Marketing dataset describe direct-marketing campaigns coducted by a Portuguese banking instituition to promote term-deposite subscriptions. The data records customer characterstics, campaign information, and whether a subscription occured.}

**Potential limitation and bias :-**
* The records come from particular bank and campaign setting, so result may not transfer to other banks or time periods.
* Some customers may appear in multiple campaign observations.
* The target records the outcome of the contact; it does not necessarily represent a customer's long-term financial needs.
* Historical campaign decisions may have affected which customers were contacted, creating selection bias.
* Some columns may be unavailable at the time the bank must choose whom to contact.

In [4]:
# Load the csv
df = pd.read_csv("bank-full.csv", sep=";")

print("Dataset Loaded Successfully!")
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

display(df.head())

Dataset Loaded Successfully!
Number of rows: 45211
Number of columns: 17


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [5]:
# Check the columns and data types 
print("Column Names:")
print(df.columns.tolist())
print("\nDatatypes:")
print(df.dtypes)
print("\nDataset Information:")
df.info()

Column Names:
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y']

Datatypes:
age          int64
job            str
marital        str
education      str
default        str
balance      int64
housing        str
loan           str
contact        str
day          int64
month          str
duration     int64
campaign     int64
pdays        int64
previous     int64
poutcome       str
y              str
dtype: object

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   age        45211 non-null  int64
 1   job        45211 non-null  str  
 2   marital    45211 non-null  str  
 3   education  45211 non-null  str  
 4   default    45211 non-null  str  
 5   balance    45211 non-null  int64
 6   housing    45211 non-null  str  
 7   l

**Source :-** From Kaggel UCI Bank Marketing dataset  
**Collection Content :-** Historical bank marketing campaigns  
**Purpose :-** Understand and improve targeting for term-deposit marketing.  
**Limitations :-** Campaign-specific data, possible repeated customers, potential selection bias, and possible leakage from post-contact information.

## **B2 — The five checks that disqualify a dataset**

In [6]:
# Check the target 
target = "y"

print("Target column exists:", target in df.columns)

print("\nTarget Values:")
print(df[target].value_counts())

print("\nTarget Percentage:")
print((df[target].value_counts(normalize=True) * 100).round(2))

Target column exists: True

Target Values:
y
no     39922
yes     5289
Name: count, dtype: int64

Target Percentage:
y
no     88.3
yes    11.7
Name: proportion, dtype: float64


In [8]:
# Check missing values and duplicates
print("Total missing values:", df.isnull().sum().sum())

print("\nMissing values by columns:")
print(df.isnull().sum().sort_values(ascending=False))

print("\nExact duplicate rows:", df.duplicated().sum())

print("\nNumber of unique values per column:")
print(df.nunique())

Total missing values: 0

Missing values by columns:
age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64

Exact duplicate rows: 0

Number of unique values per column:
age            77
job            12
marital         3
education       4
default         2
balance      7168
housing         2
loan            2
contact         3
day            31
month          12
duration     1573
campaign       48
pdays         559
previous       41
poutcome        4
y               2
dtype: int64


In [9]:
# check whether the target is defined
print("Unique target values:", df["y"].unique())

assert set(df["y"].unique()) == {"yes", "no"}

print("Target is clearly defined as yes/no.")

Unique target values: <StringArray>
['no', 'yes']
Length: 2, dtype: str
Target is clearly defined as yes/no.


**Check 1 — Is the target actually there, and is it well defined?**  
Ans. The target exists in the y column and indicates whether the customer subscribed to a term deposit. The dataset's documentation identifies this as the outcome of the marketing contact.

In [12]:
# Check dataset size 
class_count = df["y"].value_counts()

print("Total Rows:", len(df))
print("Negative Cases:", class_count["no"])
print("Positive Cases:", class_count["yes"])
print("Minority-class percentage:", round(class_count["yes"] / len(df) * 100, 2), "%")

Total Rows: 45211
Negative Cases: 39922
Positive Cases: 5289
Minority-class percentage: 11.7 %


**Check 2 — Is there enough of it?**  
Ans. The dataset has 45,211 records and 5,289 positive cases. This is sufficient for an initial classification project, although the minority class and the possibility of distribution shift need to be considered.

In [13]:
# Investigate possible leakage
print("Columns in dataset:")
for col in df.columns:
    print("-", col)

print("\nColumns requiring special attention:")
print("duration:", "Call duration; known after the call.")
print("poutcome:", "Outcome of a previous marketing campaign.")
print("pdays:", "Days since the customer was previously contacted.")
print("previous:", "Number of contacts before this campaign.")
print("day/month:", "Calendar components; no year is provided.")

Columns in dataset:
- age
- job
- marital
- education
- default
- balance
- housing
- loan
- contact
- day
- month
- duration
- campaign
- pdays
- previous
- poutcome
- y

Columns requiring special attention:
duration: Call duration; known after the call.
poutcome: Outcome of a previous marketing campaign.
pdays: Days since the customer was previously contacted.
previous: Number of contacts before this campaign.
day/month: Calendar components; no year is provided.


**Check 3 — Does a column leak the answer?**  
Ans. `duration` is the clearest leakage risk for a model intended to prioritize customers before calling them. We will exclude it. `poutcome`, `pdays`, and `previous` describe prior campaign history and may be usable if they are available at decision time. We will verify this assumption with the data owner before any real deployment.

In [14]:
# Check the Dimension
print("Date-related columns: day, month")

print("Month Values:")
print(df["month"].value_counts())

print("\nFirst 10 date components:")
display(df[['day', "month"]].head(10))

print("\nLast 10 date components:")
display(df[["day", "month"]].tail(10))

print("\nTarget rate in consecutive portions of the file:")
df["positive_target"] = (df["y"] == "yes").astype(int)

df["row_period"] = pd.qcut(np.arange(len(df)), q=5, labels=["Period 1", "Period 2", "Period 3", "Period 4", "Period 5"])

display(df.groupby("row_period", observed=True)["positive_target"]
       .agg(["count", "mean"])
       .rename(columns={"count": "rows", "mean": "positive_rate"}))

Date-related columns: day, month
Month Values:
month
may    13766
jul     6895
aug     6247
jun     5341
nov     3970
apr     2932
feb     2649
jan     1403
oct      738
sep      579
mar      477
dec      214
Name: count, dtype: int64

First 10 date components:


,day,month
0,5,may
1,5,may
2,5,may
3,5,may
4,5,may
5,5,may
6,5,may
7,5,may
8,5,may
9,5,may



Last 10 date components:


,day,month
45201,17,nov
45202,17,nov
45203,17,nov
45204,17,nov
45205,17,nov
45206,17,nov
45207,17,nov
45208,17,nov
45209,17,nov
45210,17,nov



Target rate in consecutive portions of the file:


,rows,positive_rate
row_period,,
Period 1,9043,0.033617
Period 2,9042,0.056182
Period 3,9042,0.061380
Period 4,9042,0.117894
Period 5,9042,0.315859


**Check 4 — Is there a time dimension?**  
Ans. The dataset contains day and month, but not a complete timestamp or year. The supplied records appear to follow a time-related sequence. For this assignment, we will preserve the original row order for a holdout split rather than shuffle records randomly. The absence of a year means the exact dates and duration of the holdout period cannot be independently reconstructed from these columns alone.

In [16]:
# Human predictability check
sample = df.sample(10, random_state=42)

human_check = sample.drop(
    columns=["y", "positive_target", "row_period"],
    errors="ignore"
)

print("Ten records with target hidden:")
display(human_check)

print("\nActual labels -- reveal only after attempting the guesses:")
display(sample[["y"]])

Ten records with target hidden:


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome
3776,40,blue-collar,married,secondary,no,580,yes,no,unknown,16,may,192,1,-1,0,unknown
9928,47,services,single,secondary,no,3644,no,no,unknown,9,jun,83,2,-1,0,unknown
33409,25,student,single,tertiary,no,538,yes,no,cellular,20,apr,226,1,-1,0,unknown
31885,42,management,married,tertiary,no,1773,no,no,cellular,9,apr,311,1,336,1,failure
15738,56,management,married,tertiary,no,217,no,yes,cellular,21,jul,121,2,-1,0,unknown
30813,28,blue-collar,married,secondary,no,1134,no,no,cellular,9,feb,130,3,-1,0,unknown
35463,24,management,single,tertiary,no,1085,no,yes,cellular,7,may,95,6,-1,0,unknown
31382,37,admin.,single,secondary,no,127,no,no,cellular,23,mar,83,4,-1,0,unknown
16904,30,blue-collar,single,secondary,no,3,yes,no,cellular,25,jul,51,1,-1,0,unknown
11930,38,technician,single,secondary,no,258,no,yes,unknown,20,jun,587,2,-1,0,unknown



Actual labels -- reveal only after attempting the guesses:


,y
3776,no
9928,no
33409,no
31885,no
15738,no
30813,no
35463,no
31382,no
16904,no
11930,no


**Check 5 — Can a person do this task?**  
Ans. A person can form plausible guesses using customer characteristics and previous campaign history, but cannot know the outcome with certainty. This supports trying an ML approach; it does not prove that the available features will provide useful predictive performance.